# Financial Research Workbench — five downloaded open models

Use a free T4 GPU. This notebook downloads project code from GitHub and original reports from the issuers; no browser file upload is required. Full reports stay in your runtime. No private inference API is used. LangSmith uses its free Developer plan and a separate observability key in Colab Secrets. Start with the two-question smoke set before the full showcase. Failures remain recorded failures.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from pathlib import Path
import os, sys, subprocess, venv, json, zipfile
REPOSITORY = 'https://github.com/nandevers/financial-research-workbench.git'
ROOT = Path('/content/financial-research-workbench')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1',REPOSITORY,str(ROOT)],check=True)
os.chdir(ROOT)
print('Project commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Isolate model libraries; the notebook kernel never imports Torch.
env = ROOT / 'artifacts/local/env-5.13.1'
venv.EnvBuilder(with_pip=False,system_site_packages=True).create(env)
PYTHON = str(env/'bin/python')
subprocess.run([sys.executable,'-m','pip','--python',PYTHON,'install','-q','-e','.[gpu]','transformers==5.13.1'],check=True)
# Fetch authentic issuer reports directly. Raw files are never committed.
subprocess.run([PYTHON,'-m','workbench.cli','corpus'],check=True)


In [ ]:
# Store LANGSMITH_API_KEY in Colab Secrets and enable notebook access.
# Never paste a key into notebook source, output, or git.
USE_LANGSMITH = True
if USE_LANGSMITH:
    from google.colab import userdata
    os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
    os.environ['LANGSMITH_TRACING'] = 'true'
    os.environ['LANGSMITH_PROJECT'] = 'financial-research-workbench'
else:
    os.environ['LANGSMITH_TRACING'] = 'false'


In [ ]:
CASE_SET = 'smoke'  # Banking and insurance, both cross-period comparisons.
LIMIT = 2  # For the full showcase use CASE_SET='showcase' and LIMIT=12.
MODEL = None  # Set one exact ID from models.json for an independent job.
command = [sys.executable,'scripts/colab_run.py','--limit',str(LIMIT),'--case-set',CASE_SET,'--release-cache']
if MODEL: command += ['--model',MODEL]
if USE_LANGSMITH: command += ['--langsmith']
subprocess.run(command,check=True)


In [ ]:
from google.colab import files
for p in sorted(Path('artifacts/local/models').glob('*-status.json')):
    status=json.loads(p.read_text())
    print(status['model_id'],'completed:',status['completed'],'failures:',len(status['failures']))
with zipfile.ZipFile('/content/workbench-results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('runs','models'):
        for p in Path('artifacts/local',folder).glob('*.json'):
            z.write(p,p.as_posix())
files.download('/content/workbench-results.zip')
